# Transferring Topo stuff to python

In [ ]:
# Import packages 
import os
import sys
workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature




from myPythonTools.TopoAna import Plots as Po
from myPythonTools.TopoAna import rdtilemap as Ti


import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.tri as tri
import matplotlib.colors as colors

from scipy.io import FortranFile

import importlib
import glob
import copy

importlib.reload(Po)
importlib.reload(Ti)

#sys.path.append('../Plotting/')
#   Now you can imprt modules in ../Plotting
#import xyp_plot as xyp
#import ana as a


In [ ]:
host=os.getenv('HOST')
host

## Set some paths and directories

In [ ]:
TopoRoot = '/glade/work/juliob/Topo/NCARTopoJTB-fire/'     #cases/ne30pg3_Sco100_small_rdgs/
#Case = 'ne30pg3_Sco100_small_rdgs'
#Case = 'ne30pg3_Sco50_v5_default_rdgs'  #'ne30pg3_Sco100_Rw025_v6_test'
Case = 'ne30pg3_Sco100_v7_test'
ogrid= 'ne30pg3'


foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/'+ogrid+'*.nc')
FileN = foo[-1]
print(FileN)
dCAM=xr.open_dataset( FileN )
print( list(dCAM.variables) )
print(np.shape(dCAM['lat']))
target_lon=dCAM['lon'].values
target_lat=dCAM['lat'].values
target_topo=dCAM['PHIS'].values / 9.8


foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/topo*.nc')
FileN = foo[-1]
print(FileN)
topo=xr.open_dataset( FileN )
print(list(topo.variables))
print(topo.attrs)
raw=topo['terr_dev']+topo['terr_sm']
Loo = np.size(raw)
nc=int(np.sqrt( Loo/6 ))
print(int(nc))
raw=np.reshape( raw.values , (6,nc,nc) )
dev=np.reshape( topo['terr_dev'].values , (6,nc,nc) )
smoo=np.reshape( topo['terr_sm'].values , (6,nc,nc) )
tlats=np.reshape( topo['lat'].values , (6,nc,nc) )
tlons=np.reshape( topo['lon'].values , (6,nc,nc) )

# Define the range of indices
x_indices = np.arange(3000)
y_indices = np.arange(3000)

# Use meshgrid to create 2D arrays of x and y indices
xgrid, ygrid = np.meshgrid(x_indices, y_indices, indexing='xy')


foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/Ridge_tile_map.dat')
TileFile = foo[-1]
TileMap=Ti.rdtilemap( tile=TileFile, read_angll=True )

ntarget,maxridges = TileMap.MyObject.shape
print(ntarget,maxridges)
print( f"Tile shape {TileMap.lonc.shape}" )


latc=(180./np.pi)*TileMap.latc
lonc=(180./np.pi)*TileMap.lonc
angll=TileMap.angll
anglx=TileMap.anglx
clngt=TileMap.clngt
mxdis=TileMap.mxdis



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs


# ============================================================
# 1. Choose geographical region
# ============================================================

location = 'CONUS_South'  #'Colorado'

location_info = {

    'Colorado': {
        'panel': 4,
        'lonrange': np.asarray([-117., -100.]) + 360.,
        'latrange': np.asarray([30., 41.]),
        'projection': 'LambertConformal'
    },

    'Utah': {
        'panel': 4,
        'lonrange': np.asarray([-113., -108.]) + 360.,
        'latrange': np.asarray([36., 41.]),
        'projection': 'LambertConformal'
    },

    'Kaibab': {
        'panel': 4,
        'lonrange': np.asarray([-113., -111.]) + 360.,
        'latrange': np.asarray([36., 37.]),
        'projection': 'LambertConformal'
    },

    'SouthwestUS': {
        'panel': 4,
        'lonrange': np.asarray([-130., -104.]) + 360.,
        'latrange': np.asarray([29., 42.]),
        'projection': 'AlbersEqualArea'
    },

    'CONUS_South': {
        'panel': 4,
        'lonrange': np.asarray([-125., -70.]) + 360.,
        'latrange': np.asarray([24., 45.]),
        'projection': 'AlbersEqualArea'
    },

    'SouthAndesAntarcticPen': {
        'panel': 5,
        'lonrange': np.asarray([260., 300.]),
        'latrange': np.asarray([-75., -50.]),
        'projection': 'Orthographic'
    },

    'McMurdo': {
        'panel': 5,
        'lonrange': np.asarray([120., 230.]),
        'latrange': np.asarray([-87., -50.]),
        'projection': 'Orthographic'
    },

    'Antarctica': {
        'panel': 5,
        'lonrange': np.asarray([0., 360.]),
        'latrange': np.asarray([-90., -50.]),
        'projection': 'Orthographic'
    }
}


if location not in location_info:
    raise ValueError(f"Unknown location: {location}")

cfg = location_info[location]

panel    = cfg['panel']
ip       = panel - 1
lonrange = cfg['lonrange']
latrange = cfg['latrange']

central_longitude = np.average(lonrange)
central_latitude  = np.average(latrange)


# ============================================================
# 2. Set map projection
# ============================================================

if cfg['projection'] == 'LambertConformal':

    MapProj = ccrs.LambertConformal(
        central_longitude=central_longitude,
        central_latitude=central_latitude
    )

elif cfg['projection'] == 'AlbersEqualArea':

    MapProj = ccrs.AlbersEqualArea(
        central_longitude=central_longitude,
        central_latitude=central_latitude
    )

elif cfg['projection'] == 'Orthographic':

    if location == 'Antarctica':
        central_longitude = 180.
        central_latitude  = -90.

    MapProj = ccrs.Orthographic(
        central_longitude=central_longitude,
        central_latitude=central_latitude
    )

DataProj = ccrs.PlateCarree()


# ============================================================
# 3. Find target cells in region
# ============================================================

in_region = (
    (target_lon >  lonrange[0]) &
    (target_lon <= lonrange[1]) &
    (target_lat >  latrange[0]) &
    (target_lat <= latrange[1])
)

iroo = np.where(in_region)[0]

#iroo=iroo[0:1]
print(f"Number of target cells in region = {len(iroo)}")


# ============================================================
# 4. Find tile subsection needed for plot
# ============================================================

tile_mask = (
    (tlons[ip, :, :] >  lonrange[0]) &
    (tlons[ip, :, :] <= lonrange[1]) &
    (tlats[ip, :, :] >  latrange[0]) &
    (tlats[ip, :, :] <= latrange[1])
)

if not np.any(tile_mask):
    raise ValueError("Requested region does not overlap selected topo tile")

yy_idx, xx_idx = np.where(tile_mask)

xxrange = [xx_idx.min(), xx_idx.max()]
yyrange = [yy_idx.min(), yy_idx.max()]

print("x range =", xxrange)
print("y range =", yyrange)
print("central latitude =", central_latitude)

xslice = slice(xxrange[0], xxrange[1] + 1)
yslice = slice(yyrange[0], yyrange[1] + 1)


# ============================================================
# 5. Plot settings
# ============================================================

topo_levels = np.linspace(-100., 4000., 21)
cmap = plt.get_cmap('terrain')

# Same mapping for shaded topo and ridgelet coloring
norm = mpl.colors.BoundaryNorm(
    topo_levels,
    ncolors=cmap.N,
    clip=True
)

ridge_linewidth = 1.8


# ============================================================
# 6. Helper function for one panel
# ============================================================

def plot_ridgelet_panel(ax, fill_field, panel_title, plot_ridgelets=True ):
    """
    fill_field should be either raw or smoo
    """

    # Filled topography
    cf = ax.contourf(
        tlons[ip, yslice, xslice],
        tlats[ip, yslice, xslice],
        fill_field[ip, yslice, xslice],
        levels=topo_levels,
        cmap=cmap,
        norm=norm,
        alpha=0.5,
        transform=DataProj
    )

    # Smoothed-topography contours
    ax.contour(
        tlons[ip, yslice, xslice],
        tlats[ip, yslice, xslice],
        smoo[ip, yslice, xslice],
        levels=topo_levels,
        cmap=cmap,
        norm=norm,
        alpha=0.5,
        linewidths=0.7,
        transform=DataProj
    )
    
    # Ridgelets colored by mxdis + local smoothed target_topo
    if plot_ridgelets == True:
        for i in iroo:
            for ir in range(maxridges):
    
                value = mxdis[i, ir] + target_topo[i]
    
                if not np.all(np.isfinite([
                    latc[i, ir],
                    lonc[i, ir],
                    angll[i, ir],
                    clngt[i, ir],
                    value
                ])):
                    continue
                    
                
                lat0 = latc[i, ir]
                lon0 = lonc[i, ir]
    
                lat1, lon1, lat2, lon2 = Po.CAMridgelet(
                    lat=lat0,
                    lon=lon0,
                    angll=angll[i, ir],
                    clngt=clngt[i, ir]
                )
    
                ridge_color = cmap(norm(value))
    
                ax.plot(
                    [lon1, lon2],
                    [lat1, lat2],
                    color=ridge_color,
                    linewidth=ridge_linewidth,
                    transform=DataProj,
                    zorder=10
                )
    
    ax.set_extent(
        [lonrange[0], lonrange[1], latrange[0], latrange[1]],
        crs=DataProj
    )

    gl = ax.gridlines(
        draw_labels=True,
        linewidth=1,
        color='gray',
        alpha=0.5,
        linestyle='--'
    )

    gl.top_labels   = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 11, 'color': 'gray'}
    gl.ylabel_style = {'size': 11, 'color': 'gray'}

    ax.set_title(panel_title, fontsize=13)

    return cf


# ============================================================
# 7. Create 2-panel figure
# ============================================================

fig = plt.figure(figsize=(18, 9))

ax1 = fig.add_axes([0.05, 0.12, 0.40, 0.78], projection=MapProj)
ax2 = fig.add_axes([0.52, 0.12, 0.40, 0.78], projection=MapProj)

cf1 = plot_ridgelet_panel(
    ax1,
    fill_field=raw,
    plot_ridgelets=False,
    panel_title="Raw topography shading + smoothed contours\nRidgelet color = mxdis + target_topo"
)

cf2 = plot_ridgelet_panel(
    ax2,
    fill_field=smoo,
    panel_title="Smoothed topography shading + smoothed contours\nRidgelet color = mxdis + target_topo"
)


# ============================================================
# 8. Shared colorbar
# ============================================================

cax = fig.add_axes([0.94, 0.18, 0.02, 0.62])

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, cax=cax)
cbar.set_label('Elevation-like value (m)', fontsize=12)
# If you prefer to be more explicit:
# cbar.set_label('mxdis + target_topo (m)', fontsize=12)


# ============================================================
# 9. Suptitle
# ============================================================

fig.suptitle(
    "3000x3000 topography over-plotted with ridgelets\n"
    f"{TileFile}",
    fontsize=15,
    y=0.97
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs


# ============================================================
# 1. Choose geographical region
# ============================================================

location = 'CONUS_South'  #'Colorado'

location_info = {

    'Colorado': {
        'panel': 4,
        'lonrange': np.asarray([-117., -100.]) + 360.,
        'latrange': np.asarray([30., 41.]),
        'projection': 'LambertConformal'
    },

    'Utah': {
        'panel': 4,
        'lonrange': np.asarray([-113., -108.]) + 360.,
        'latrange': np.asarray([36., 41.]),
        'projection': 'LambertConformal'
    },

    'Kaibab': {
        'panel': 4,
        'lonrange': np.asarray([-113., -111.]) + 360.,
        'latrange': np.asarray([36., 37.]),
        'projection': 'LambertConformal'
    },

    'SouthwestUS': {
        'panel': 4,
        'lonrange': np.asarray([-130., -104.]) + 360.,
        'latrange': np.asarray([29., 42.]),
        'projection': 'AlbersEqualArea'
    },

    'CONUS_South': {
        'panel': 4,
        'lonrange': np.asarray([-125., -70.]) + 360.,
        'latrange': np.asarray([24., 45.]),
        'projection': 'AlbersEqualArea'
    },

    'SouthAndesAntarcticPen': {
        'panel': 5,
        'lonrange': np.asarray([260., 300.]),
        'latrange': np.asarray([-75., -50.]),
        'projection': 'Orthographic'
    },

    'McMurdo': {
        'panel': 5,
        'lonrange': np.asarray([120., 230.]),
        'latrange': np.asarray([-87., -50.]),
        'projection': 'Orthographic'
    },

    'Antarctica': {
        'panel': 5,
        'lonrange': np.asarray([0., 360.]),
        'latrange': np.asarray([-90., -50.]),
        'projection': 'Orthographic'
    }
}


if location not in location_info:
    raise ValueError(f"Unknown location: {location}")

cfg = location_info[location]

panel    = cfg['panel']
ip       = panel - 1
lonrange = cfg['lonrange']
latrange = cfg['latrange']

central_longitude = np.average(lonrange)
central_latitude  = np.average(latrange)


# ============================================================
# 2. Set map projection
# ============================================================

if cfg['projection'] == 'LambertConformal':

    MapProj = ccrs.LambertConformal(
        central_longitude=central_longitude,
        central_latitude=central_latitude
    )

elif cfg['projection'] == 'AlbersEqualArea':

    MapProj = ccrs.AlbersEqualArea(
        central_longitude=central_longitude,
        central_latitude=central_latitude
    )

elif cfg['projection'] == 'Orthographic':

    if location == 'Antarctica':
        central_longitude = 180.
        central_latitude  = -90.

    MapProj = ccrs.Orthographic(
        central_longitude=central_longitude,
        central_latitude=central_latitude
    )

DataProj = ccrs.PlateCarree()


# ============================================================
# 3. Find target cells in region
# ============================================================

in_region = (
    (target_lon >  lonrange[0]) &
    (target_lon <= lonrange[1]) &
    (target_lat >  latrange[0]) &
    (target_lat <= latrange[1])
)

iroo = np.where(in_region)[0]

#iroo=iroo[0:1]
print(f"Number of target cells in region = {len(iroo)}")


# ============================================================
# 4. Find tile subsection needed for plot
# ============================================================

tile_mask = (
    (tlons[ip, :, :] >  lonrange[0]) &
    (tlons[ip, :, :] <= lonrange[1]) &
    (tlats[ip, :, :] >  latrange[0]) &
    (tlats[ip, :, :] <= latrange[1])
)

if not np.any(tile_mask):
    raise ValueError("Requested region does not overlap selected topo tile")

yy_idx, xx_idx = np.where(tile_mask)

xxrange = [xx_idx.min(), xx_idx.max()]
yyrange = [yy_idx.min(), yy_idx.max()]

print("x range =", xxrange)
print("y range =", yyrange)
print("central latitude =", central_latitude)

xslice = slice(xxrange[0], xxrange[1] + 1)
yslice = slice(yyrange[0], yyrange[1] + 1)


# ============================================================
# 5. Plot settings
# ============================================================

topo_levels = np.linspace(-1000., 1000., 21)
cmap = plt.get_cmap('terrain')

# Same mapping for shaded topo and ridgelet coloring
norm = mpl.colors.BoundaryNorm(
    topo_levels,
    ncolors=cmap.N,
    clip=True
)

ridge_linewidth = 1.8


# ============================================================
# 6. Helper function for one panel
# ============================================================

def plot_ridgelet_panel(ax, fill_field, panel_title, plot_ridgelets=True ):
    """
    fill_field should be either raw or smoo
    """

    # Filled topography
    cf = ax.contourf(
        tlons[ip, yslice, xslice],
        tlats[ip, yslice, xslice],
        fill_field[ip, yslice, xslice],
        levels=topo_levels,
        cmap=cmap,
        norm=norm,
        alpha=0.5,
        transform=DataProj
    )
    
    # Ridgelets colored by mxdis + local smoothed target_topo
    if plot_ridgelets == True:
        for i in iroo:
            for ir in range(maxridges):
    
                value = mxdis[i, ir] 
    
                if not np.all(np.isfinite([
                    latc[i, ir],
                    lonc[i, ir],
                    angll[i, ir],
                    clngt[i, ir],
                    value
                ])):
                    continue
                    
                
                lat0 = latc[i, ir]
                lon0 = lonc[i, ir]
    
                lat1, lon1, lat2, lon2 = Po.CAMridgelet(
                    lat=lat0,
                    lon=lon0,
                    angll=angll[i, ir],
                    clngt=clngt[i, ir]
                )
    
                ridge_color = cmap(norm(value))
    
                ax.plot(
                    [lon1, lon2],
                    [lat1, lat2],
                    color=ridge_color,
                    linewidth=ridge_linewidth,
                    transform=DataProj,
                    zorder=10
                )
    
    ax.set_extent(
        [lonrange[0], lonrange[1], latrange[0], latrange[1]],
        crs=DataProj
    )

    gl = ax.gridlines(
        draw_labels=True,
        linewidth=1,
        color='gray',
        alpha=0.5,
        linestyle='--'
    )

    gl.top_labels   = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 11, 'color': 'gray'}
    gl.ylabel_style = {'size': 11, 'color': 'gray'}

    ax.set_title(panel_title, fontsize=13)

    return cf


# ============================================================
# 7. Create 2-panel figure
# ============================================================

fig = plt.figure(figsize=(18, 9))

ax1 = fig.add_axes([0.05, 0.12, 0.40, 0.78], projection=MapProj)
ax2 = fig.add_axes([0.52, 0.12, 0.40, 0.78], projection=MapProj)

cf1 = plot_ridgelet_panel(
    ax1,
    fill_field=dev,
    plot_ridgelets=False,
    panel_title="Dev topography shading \nRidgelet color = mxdis + target_topo"
)

cf2 = plot_ridgelet_panel(
    ax2,
    fill_field=dev,
    panel_title="Dev topography shading \nRidgelet color = mxdis + target_topo"
)


# ============================================================
# 8. Shared colorbar
# ============================================================

cax = fig.add_axes([0.94, 0.18, 0.02, 0.62])

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, cax=cax)
cbar.set_label('Elevation-like value (m)', fontsize=12)
# If you prefer to be more explicit:
# cbar.set_label('mxdis + target_topo (m)', fontsize=12)


# ============================================================
# 9. Suptitle
# ============================================================

fig.suptitle(
    "3000x3000 topography over-plotted with ridgelets\n"
    f"{TileFile}",
    fontsize=15,
    y=0.97
)

plt.show()

In [ ]:
print( TileMap.MyObject[iroo,0:] )
print( TileMap.WtObject[iroo,0:] )
print( TileMap.mxdis[iroo,0:] )


#print( lonc[iroo[0],0], latc[iroo[0],0] )
#print( lonc[iroo[1],0], latc[iroo[1],0] )


In [ ]:
#import numpy as np
#import matplotlib.pyplot as plt
import matplotlib as mpl
#import cartopy.crs as ccrs


# ============================================================
# 1. Choose geographical region
# ============================================================

location = 'Colorado'

location_info = {

    'Colorado': {
        'panel': 4,
        'lonrange': np.asarray([-117., -100.]) + 360.,
        'latrange': np.asarray([30., 41.]),
        'projection': 'LambertConformal'
    },

    'SouthwestUS': {
        'panel': 4,
        'lonrange': np.asarray([-130., -104.]) + 360.,
        'latrange': np.asarray([29., 42.]),
        'projection': 'AlbersEqualArea'
    },

    'SouthAndesAntarcticPen': {
        'panel': 5,
        'lonrange': np.asarray([260., 300.]),
        'latrange': np.asarray([-75., -50.]),
        'projection': 'Orthographic'
    },

    'McMurdo': {
        'panel': 5,
        'lonrange': np.asarray([120., 230.]),
        'latrange': np.asarray([-87., -50.]),
        'projection': 'Orthographic'
    },

    'Antarctica': {
        'panel': 5,
        'lonrange': np.asarray([0., 360.]),
        'latrange': np.asarray([-90., -50.]),
        'projection': 'Orthographic'
    }
}


if location not in location_info:
    raise ValueError(f"Unknown location: {location}")

cfg = location_info[location]

panel    = cfg['panel']
ip       = panel - 1
lonrange = cfg['lonrange']
latrange = cfg['latrange']

central_longitude = np.average(lonrange)
central_latitude  = np.average(latrange)


# ============================================================
# 2. Set map projection
# ============================================================

if cfg['projection'] == 'LambertConformal':

    MapProj = ccrs.LambertConformal(
        central_longitude=central_longitude,
        central_latitude=central_latitude
    )

elif cfg['projection'] == 'AlbersEqualArea':

    MapProj = ccrs.AlbersEqualArea(
        central_longitude=central_longitude,
        central_latitude=central_latitude
    )

elif cfg['projection'] == 'Orthographic':

    # Special case for whole Antarctica
    if location == 'Antarctica':
        central_longitude = 180.
        central_latitude  = -90.

    MapProj = ccrs.Orthographic(
        central_longitude=central_longitude,
        central_latitude=central_latitude
    )


DataProj = ccrs.PlateCarree()


# ============================================================
# 3. Find target grid cells containing ridgelets in region
# ============================================================

in_region = (
    (target_lon >  lonrange[0]) &
    (target_lon <= lonrange[1]) &
    (target_lat >  latrange[0]) &
    (target_lat <= latrange[1])
)

iroo = np.where(in_region)[0]

print(f"Number of target cells in region = {len(iroo)}")


# ============================================================
# 4. Find raw-topography tile subsection needed for plot
# ============================================================

tile_mask = (
    (tlons[ip, :, :] >  lonrange[0]) &
    (tlons[ip, :, :] <= lonrange[1]) &
    (tlats[ip, :, :] >  latrange[0]) &
    (tlats[ip, :, :] <= latrange[1])
)

if not np.any(tile_mask):
    raise ValueError("Requested region does not overlap selected topo tile")


yy_idx, xx_idx = np.where(tile_mask)

xxrange = [xx_idx.min(), xx_idx.max()]
yyrange = [yy_idx.min(), yy_idx.max()]

print("x range =", xxrange)
print("y range =", yyrange)
print("central latitude =", central_latitude)


# Include upper endpoint in Python slices
xslice = slice(xxrange[0], xxrange[1] + 1)
yslice = slice(yyrange[0], yyrange[1] + 1)


# ============================================================
# 5. Plot settings
# ============================================================

topo_levels = np.linspace(-100., 4000., 21)
cyc_levels = np.linspace( 0, 20, 20 )
cmap = plt.get_cmap('terrain')
cmap_cyc = plt.get_cmap('tab20')

# Use exactly the same mapping for topography and mxdis
norm = mpl.colors.BoundaryNorm(
    topo_levels,
    ncolors=cmap.N,
    clip=True
)
norm_cyc = mpl.colors.BoundaryNorm(
    cyc_levels,
    ncolors=cmap_cyc.N,
    clip=True
)

ridge_linewidth = 1.8


# ============================================================
# 6. Create figure and plot topography
# ============================================================

fig = plt.figure(figsize=(15, 15))

ax = fig.add_axes(
    [.1, .1, .8, .8],
    projection=MapProj
)


# Raw 3-km topography
cf = ax.contourf(
    tlons[ip, yslice, xslice],
    tlats[ip, yslice, xslice],
    smoo[ip, yslice, xslice],
    levels=topo_levels,
    cmap=cmap,
    norm=norm,
    alpha=0.5,
    transform=DataProj
)


# Smoothed topography
ax.contour(
    tlons[ip, yslice, xslice],
    tlats[ip, yslice, xslice],
    smoo[ip, yslice, xslice],
    levels=topo_levels,
    cmap=cmap,
    norm=norm,
    alpha=0.5,
    linewidths=0.7,
    transform=DataProj
)


# ============================================================
# 7. Plot ridgelets, color-coded by mxdis
# ============================================================

for i in iroo:

    for ir in range(maxridges):

        value = mxdis[i, ir] + target_topo[i]
        value = TileMap.MyObject[ i, ir ] % 20

        # Skip unusable ridgelets
        if not np.all(np.isfinite([
            latc[i, ir],
            lonc[i, ir],
            angll[i, ir],
            clngt[i, ir],
            value
        ])):
            continue

        if TileMap.MyObject[ i, ir ] < 0:
            continue

        lat0 = latc[i, ir]
        lon0 = lonc[i, ir]

        lat1, lon1, lat2, lon2 = Po.CAMridgelet(
            lat=lat0,
            lon=lon0,
            angll=angll[i, ir],
            clngt=clngt[i, ir]
        )

        #ridge_color = cmap(norm(value))
        ridge_color = cmap_cyc(norm_cyc(value))

        ax.plot(
            [lon1, lon2],
            [lat1, lat2],
            color=ridge_color,
            linewidth=ridge_linewidth,
            transform=DataProj,
            zorder=10
        )


# ============================================================
# 8. Map extent and grid lines
# ============================================================

ax.set_extent(
    [lonrange[0], lonrange[1],
     latrange[0], latrange[1]],
    crs=DataProj
)


gl = ax.gridlines(
    draw_labels=True,
    linewidth=1,
    color='gray',
    alpha=0.5,
    linestyle='--'
)

gl.top_labels   = False
gl.right_labels = False

gl.xlabel_style = {'size': 12, 'color': 'gray'}
gl.ylabel_style = {'size': 12, 'color': 'gray'}


# ============================================================
# 9. Colorbar for ridgelet mxdis
# ============================================================

# Independent ScalarMappable makes it explicit that the colorbar
# refers to mxdis rather than the shaded topography.
ridge_sm = mpl.cm.ScalarMappable(
    norm=norm,
    cmap=cmap
)

ridge_sm.set_array([])

cbar = fig.colorbar(
    ridge_sm,
    ax=ax,
    orientation='vertical',
    shrink=0.65,
    pad=0.03
)

cbar.set_label('mxdis (m)', fontsize=13)


# ============================================================
# 10. Title
# ============================================================

ax.set_title(
    "Raw 3000x3000 topography over-plotted with ridgelets\n"
    "Ridgelet color = mxdis\n"
    f"{TileFile}",
    fontsize=14
)

plt.show()

In [ ]:
TileMap


In [ ]:
from dataclasses import fields
[f.name for f in fields(TileMap)]

In [ ]:
plt.plot( TileMap.MyObject[iroo,0] )
plt.ylim(250000,270000)

In [ ]:
10022 % 10000
